# Using a Remote MCP Server as Tools (Web Search)

### Introduction

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) lets you expose tools via a standard protocol that any MCP-compatible client can consume. This is especially powerful for **remote hosted MCP servers** — tools running as HTTP services that your agent can call over the network.

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools, so our agent can perform live web searches.

GitHub page for Open Web Search: https://github.com/Aas-ee/open-webSearch

### Prerequisites

- Completion of steps in `010_langchain_basics.ipynb` where we deployed an Azure Foundry with an LLM model and a web search MCP server deployed on Azure Functions Apps (see `function_mcp_server_web_search.tf`).

### Getting the LLM Endpoint and API Key

To use the LLM model with `langchain`, we first need to retrieve the endpoint, API key and the MCP server endpoint from the Terraform outputs. You can do this by running the following command.


In [1]:
foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

app_service_mcp_server_open_web_search_fqdn = ! terraform -chdir=infra output -raw app_service_mcp_server_open_web_search_fqdn
app_service_mcp_server_open_web_search_fqdn = app_service_mcp_server_open_web_search_fqdn.n
print("MCP Server Endpoint (App Service):", app_service_mcp_server_open_web_search_fqdn)

Foundry Endpoint: https://foundry-400.cognitiveservices.azure.com/
Foundry API Key: AAACOGYQ3t...
LLM Model Deployment Name (ChatGPT): gpt-5.4
MCP Server Endpoint (App Service): mcp-open-web-search.azurewebsites.net


### 1. Simple Chat - No Tools

Then we can test the endpoint and key by making a simple API call to the LLM model.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Connect to the LLM endpoint hosted on Foundry
model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    max_completion_tokens=512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- planning, organizing, and problem-solving

I don’t have feelings, personal experiences, or a physical body, and I don’t “know” things the way humans do. I generate responses by recognizing patterns in data I was trained on and by using the context of our conversation.

A few useful things to know:
- I can be fast and flexible across many topics.
- I can make mistakes, so it’s good to double-check important facts.
- I don’t automatically know real-time information unless it’s provided or I have access to tools that supply it.

If you want, I can also tell you:
- what I’m good at
- what my limitations are
- how to get better answers from me

### 2. Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes and makes them available as LangChain tools.


In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"https://{app_service_mcp_server_open_web_search_fqdn}/mcp",
            # "url": f"https://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### 3. Invoke a Tool Directly

Before handing the tools to an agent, we can call one of them directly to verify the connection works. Here we invoke the `search` tool and inspect the raw result returned by the MCP server.


In [5]:
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke(
    {
        "query": "What are the latest LLM models?", 
        "limit": 20, 
        "engines": ["duckduckgo"]
    })

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=4))

{
    "query": "What are the latest LLM models?",
    "engines": [
        "duckduckgo"
    ],
    "totalResults": 20,
    "results": [
        {
            "title": "AI Leaderboard 2026: Compare &amp; Rank 300+ Top AI Models by Intelligence ...",
            "url": "https://llm-stats.com/",
            "description": "<b>The</b> AI Leaderboard \u2014 independent rankings of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI <b>models</b> by intelligence, speed and price. Composite <b>LLM</b> Stats Score updated continuously from public benchmarks and live API metrics.",
            "source": "llm-stats.com",
            "engine": "duckduckgo"
        },
        {
            "title": "Best LLM Leaderboard 2026 | AI Model Rankings, Benchmarks &amp; Pricing",
            "url": "https://onyx.app/llm-leaderboard",
            "description": "<b>The</b> definitive <b>LLM</b> leaderboard \u2014 ranking the best AI <b>models</b> including Claude, GPT, Gemini, DeepSeek, Llama, and more across

### 4. Error Handling for MCP Servers

Remote tools can fail due to network issues or timeouts. Use **interceptors** to wrap tool execution with retry logic and fallback values, making the agent more resilient.


In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."


### 5. Create the MCP Client with Interceptors

Re-create the `MultiServerMCPClient`, this time registering the `retry_interceptor` and `fallback_interceptor` so every tool call is automatically protected.


In [7]:
# Connect to the remote MCP server over Streamable HTTP
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{app_service_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    },
    tool_interceptors=[retry_interceptor, fallback_interceptor]
)

### 6. Add a Custom Tool — Fetch Webpage Content

In addition to the MCP search tool, we define a local `@tool` that fetches a webpage and converts its HTML to markdown. The agent can combine both: search the web for relevant URLs, then fetch and read their content.


In [8]:
import httpx
from langchain.tools import tool
from markdownify import markdownify

@tool
def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch webpage and convert HTML to markdown."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
    
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching {url}: {e!s}"

### 7. Create and Run the Agent

Use `create_agent` to build a ReAct agent with the remote `search` tool and the local `fetch_webpage_content` tool. The agent will reason about when to search the web, fetch pages, and synthesize a final answer with citations.


In [9]:
from IPython.display import Markdown, display

search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

agent_with_mcp = create_agent(model=model, tools=[search_tool, fetch_webpage_content])

last_message = None

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content=
    """
        What are the latest open and close LLM models in 2026 ?
        Compare their capabilities and limitations.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    last_message = step["messages"][-1]

# show the response content as markdown
content = getattr(last_message, "content", "")
if isinstance(content, list):
    markdown_text = "\n".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )
else:
    markdown_text = str(content)

display(Markdown(markdown_text))

================================ Human Message =================================


        What are the latest open and close LLM models in 2026 ?
        Compare their capabilities and limitations.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_Yb1MaJhoVMpjNDGo0ihyeHbt)
 Call ID: call_Yb1MaJhoVMpjNDGo0ihyeHbt
  Args:
    query: latest open source and closed source LLM models 2026 capabilities limitations 2026 model release official
    limit: 50
    searchMode: request
    engines: ['startpage', 'duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest open source and closed source LLM models 2026 capabilities limitations 2026 model release official",\n  "engines": [\n    "startpage",\n    "duckduckgo"\n  ]

I searched the web and fetched multiple pages. One requested source hit rate limits, and the search engine returned 25 results rather than 50, so I used the highest-signal fetched pages plus official/benchmark-style sources available.

## Short answer

In 2026, the **closed/frontier** LLM landscape is led by families such as:

- **OpenAI GPT-5 / GPT-5.x**
- **Anthropic Claude 4.5 / Opus-class**
- **Google Gemini 3 Pro / 2.5–3 series**
- **xAI Grok-4 Heavy**
- Possibly newer entrants like **MiniMax M-series** in some benchmark reports, though evidence is less standardized than for the big labs.

The **open/open-weight** landscape is led by:

- **GLM-5.1 / GLM-4.6 / GLM-4.7**
- **Kimi K2.x**
- **Qwen3 / Qwen3.5**
- **DeepSeek V4 / V3.2 / R1 family**
- **Llama 4**
- **Gemma 4 / Gemma 3**
- **Mistral Large 3 / Small 3.x**
- **Phi-4** for smaller local reasoning

---

# Open vs closed: what “latest” means in 2026

A key nuance: many “open-source LLM” articles in 2026 are really discussing **open-weight** models, not fully open-source in the OSI sense.  
The Hugging Face article explicitly notes that most popular models are **open-weight**, not fully open source, because training data/pipeline are not fully public. This matters for auditability, retraining, and legal clarity.

---

# Latest closed models in 2026

## 1. OpenAI GPT-5 family
### Strengths
- Very strong general reasoning and broad benchmark performance
- Strong coding, especially on newer coding benchmarks and agentic workflows
- Mature ecosystem, tooling, deployment pathways
- Strong multimodality in OpenAI system stack

### Limitations
- Closed weights; users depend on API access and pricing
- Limited transparency compared with open-weight alternatives
- Harder to audit or self-host
- Enterprise lock-in risk

## 2. Anthropic Claude 4.5 / Opus-class
### Strengths
- Often considered top-tier for writing, long-form synthesis, and agentic coding
- Strong

### More Resources

- [LangChain MCP Adapters](https://github.com/langchain-ai/langchain-mcp-adapters) — connect LangChain agents to MCP servers
- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) — the open standard for tool interoperability
- [Open Web Search MCP Server](https://github.com/Aas-ee/open-webSearch) — the web search MCP server used in this notebook
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
- [ChatOpenAI with custom endpoints](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)
